In [12]:
!pip install requests pandas psycopg2-binary -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import requests
import pandas as pd
import psycopg2
from typing import List, Dict
def get_all_paginated_data(url: str, params: Dict = None) -> List[Dict]:

    all_data = []
    page = 1
    while True:
        if params:
            current_params = params.copy()
            current_params['page'] = page
        else:
            current_params = {'page': page}

        try:
            response = requests.get(url, params=current_params)
            response.raise_for_status()

            data = response.json()

            if not data:
                break

            all_data.extend(data)
            print(f"Получена страница {page}: {len(data)} записей")

            if len(data) < 10:
                break

            page += 1

        except requests.exceptions.RequestException as e:
            print(f"Ошибка при запросе страницы {page}: {e}")
            break

    return all_data

print("Получение информации о книгах")
books_url = "https://www.anapioficeandfire.com/api/books"
all_books = get_all_paginated_data(books_url)
df_books = pd.DataFrame(all_books)
print(f"Всего книг получено: {len(df_books)}")
print(df_books[['name','released']])
df_books.to_csv('books.csv', index=False, encoding='utf-8')


Получение информации о книгах
Получена страница 1: 10 записей
Получена страница 2: 2 записей
Всего книг получено: 12
                              name             released
0                A Game of Thrones  1996-08-01T00:00:00
1                 A Clash of Kings  1999-02-02T00:00:00
2                A Storm of Swords  2000-10-31T00:00:00
3                 The Hedge Knight  2005-03-09T00:00:00
4                A Feast for Crows  2005-11-08T00:00:00
5                  The Sworn Sword  2008-06-18T00:00:00
6               The Mystery Knight  2011-03-29T00:00:00
7             A Dance with Dragons  2011-07-12T00:00:00
8       The Princess and the Queen  2013-12-03T00:00:00
9                 The Rogue Prince  2014-06-17T00:00:00
10       The World of Ice and Fire  2014-10-28T00:00:00
11  A Knight of the Seven Kingdoms  2015-10-06T00:00:00


In [14]:
print("Получение всех домов вестероса")
houses_url = "https://www.anapioficeandfire.com/api/houses"
all_houses = get_all_paginated_data(houses_url)
df_houses = pd.DataFrame(all_houses)
print(f"Всего домов получено: {len(df_houses)}")
df_houses.to_csv('all_houses.csv', index=False, encoding='utf-8')

Получение всех домов вестероса
Получена страница 1: 10 записей
Получена страница 2: 10 записей
Получена страница 3: 10 записей
Получена страница 4: 10 записей
Получена страница 5: 10 записей
Получена страница 6: 10 записей
Получена страница 7: 10 записей
Получена страница 8: 10 записей
Получена страница 9: 10 записей
Получена страница 10: 10 записей
Получена страница 11: 10 записей
Получена страница 12: 10 записей
Получена страница 13: 10 записей
Получена страница 14: 10 записей
Получена страница 15: 10 записей
Получена страница 16: 10 записей
Получена страница 17: 10 записей
Получена страница 18: 10 записей
Получена страница 19: 10 записей
Получена страница 20: 10 записей
Получена страница 21: 10 записей
Получена страница 22: 10 записей
Получена страница 23: 10 записей
Получена страница 24: 10 записей
Получена страница 25: 10 записей
Получена страница 26: 10 записей
Получена страница 27: 10 записей
Получена страница 28: 10 записей
Получена страница 29: 10 записей
Получена страница 30:

In [15]:
print("Получение домов с девизом")
params_with_words = {'hasWords': 'true'}
houses_with_words = get_all_paginated_data(houses_url, params_with_words)
df_houses_with_words = pd.DataFrame(houses_with_words)
print(f"Домов с девизом получено: {len(df_houses_with_words)}")
df_houses_with_words.to_csv('houses_with_deviz.csv', index=False, encoding='utf-8')

Получение домов с девизом
Получена страница 1: 10 записей
Получена страница 2: 10 записей
Получена страница 3: 10 записей
Получена страница 4: 10 записей
Получена страница 5: 10 записей
Получена страница 6: 10 записей
Получена страница 7: 8 записей
Домов с девизом получено: 68


In [16]:
print("РАБОТА С БАЗОЙ ДАННЫХ")

try:
    conn_params = {
        'host': 'hh-pgsql-public.ebi.ac.uk',
        'database': 'pfmegrnargs',
        'user': 'reader',
        'password': 'NWDMCE5xdipIjRrp',
        'port': 5432
    }

    print("Подключение к базе данных")
    conn = psycopg2.connect(**conn_params)

    cursor = conn.cursor()

    print("Получение 10 строк из таблицы rnc_database")
    cursor.execute("SELECT * FROM rnc_database LIMIT 10")
    rows = cursor.fetchall()

    column_names = [desc[0] for desc in cursor.description]

    df_rnc = pd.DataFrame(rows, columns=column_names)
    print(f"Данные сохранены в DataFrame. Форма: {df_rnc.shape}")
    print("строки из таблицы rnc_database:")
    print(df_rnc)

    print("Получение конкретных столбцов для 10 строк")
    cursor.execute("""
        SELECT display_name, num_sequences, num_organisms, url
        FROM rnc_database
        LIMIT 10
    """)
    selected_rows = cursor.fetchall()

    df_rnc_selected = pd.DataFrame(
        selected_rows,
        columns=['display_name', 'num_sequences', 'num_organisms', 'url']
    )
    print("Выбранные столбцы сохранены в DataFrame:")
    print(df_rnc_selected)
    cursor.close()
    conn.close()
    print("Соединение закрыто.")

except Exception as e:
    print(f"Ошибка: {e}")
    import traceback

    traceback.print_exc()

РАБОТА С БАЗОЙ ДАННЫХ
Подключение к базе данных
Получение 10 строк из таблицы rnc_database
Данные сохранены в DataFrame. Форма: (10, 19)
строки из таблицы rnc_database:
   id  timestamp userstamp      descr  current_release full_descr alive  \
0  21 2017-05-02    RNACEN    NONCODE              146    NONCODE     Y   
1   5 2017-05-17    RNACEN       VEGA               98       VEGA     N   
2  26 2017-05-01    RNACEN    GENCODE              450    GENCODE     N   
3   1 2017-05-01    RNACEN        ENA              968        ENA     Y   
4  14 2017-05-01    RNACEN       TAIR              982       TAIR     Y   
5   9 2017-05-01    RNACEN     REFSEQ              969     RefSeq     Y   
6  41 2017-05-01    RNACEN  GENECARDS              978  MalaCards     Y   
7  10 2017-05-01    RNACEN        RDP               85        RDP     Y   
8  20 2017-05-01    RNACEN  LNCIPEDIA              935  LNCipedia     Y   
9  15 2017-05-02    RNACEN   WORMBASE              970   WormBase     Y   

  for